In [ ]:
# TalTech semantic pipeline — Notebook 04
# Memory-safe SPARQL traceability using PyOxigraph
# Corrected local version for:
# Repository root (run Jupyter from this directory)

from pathlib import Path
import csv
import shutil
import time
import pandas as pd

# ============================================================
# 0. LOCAL PROJECT PATHS
# ============================================================

PROJECT_DIR = Path.cwd().resolve()
OUTPUT_DIR = PROJECT_DIR / "outputs"
RDF_DIR = OUTPUT_DIR / "03_rdf"
OUT = OUTPUT_DIR / "04_sparql"
STORE_DIR = OUTPUT_DIR / "04_oxigraph_store"

GRAPH_FILE = RDF_DIR / "taltech_native_semantic_graph.ttl"
LOAD_MARKER = STORE_DIR / ".taltech_loaded"

OUT.mkdir(parents=True, exist_ok=True)

assert PROJECT_DIR.exists(), f"Project directory not found: {PROJECT_DIR}"
assert RDF_DIR.exists(), f"RDF directory not found: {RDF_DIR}"
assert GRAPH_FILE.exists(), f"RDF graph not found: {GRAPH_FILE}"

try:
    from pyoxigraph import Store, RdfFormat
except ImportError:
    raise ImportError(
        "\nPyOxigraph is required.\n"
        "Install it once in Jupyter with:\n\n"
        "    %pip install pyoxigraph\n\n"
        "Then restart the kernel and run this script again.\n"
    )

print("Project directory :", PROJECT_DIR)
print("RDF graph         :", GRAPH_FILE)
print("Output directory  :", OUT)
print("Oxigraph store    :", STORE_DIR)
print(f"RDF file size     : {GRAPH_FILE.stat().st_size / (1024**2):.2f} MB")

# ============================================================
# 1. USE A FRESH OXIGRAPH STORE
# ============================================================

# A new store directory is used deliberately to avoid any stale
# LOCK file or Store object left open by a previous Jupyter run.
STORE_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# 2. OPEN PERSISTENT OXIGRAPH STORE
# ============================================================

store = Store(str(STORE_DIR))

# ============================================================
# 3. LOAD RDF ONLY ONCE
# ============================================================

if not LOAD_MARKER.exists():

    print("\n====================================")
    print("LOADING RDF INTO OXIGRAPH")
    print("====================================")

    start = time.perf_counter()

    # IMPORTANT:
    # `path=` tells PyOxigraph that this is a file path.
    # `bulk_load` is preferred here for this large Turtle file.
    store.bulk_load(
        path=str(GRAPH_FILE),
        format=RdfFormat.TURTLE
    )

    load_seconds = time.perf_counter() - start

    # Marker is created ONLY after successful import.
    LOAD_MARKER.write_text(
        "TalTech native RDF graph successfully loaded",
        encoding="utf-8"
    )

    print(f"✓ RDF loaded into disk store in {load_seconds:.2f} s")

else:
    load_seconds = None
    print("\n✓ Existing Oxigraph disk store reused")

# ============================================================
# 4. HELPERS
# ============================================================

query_timings = []

def term_to_python(term):
    if term is None:
        return None

    value = getattr(term, "value", None)
    if value is not None:
        return value

    return str(term)

def scalar_count(name, query):
    start = time.perf_counter()

    result = store.query(query)
    row = next(iter(result))

    elapsed = time.perf_counter() - start
    value = int(term_to_python(row[0]))

    query_timings.append({
        "query": name,
        "seconds": elapsed,
        "rows_returned": 1
    })

    print(f"{name}: {value:,} | {elapsed:.3f} s")

    return value

# ============================================================
# 5. COMPACT SPARQL VALIDATION
# ============================================================

Q_OBS = """
PREFIX sosa: <http://www.w3.org/ns/sosa/>
SELECT (COUNT(?o) AS ?count)
WHERE {
    ?o a sosa:Observation .
}
"""

Q_DEVICES = """
PREFIX ex: <https://example.org/taltech-edt/>
SELECT (COUNT(DISTINCT ?device) AS ?count)
WHERE {
    ?device a ex:OperationalDevice .
}
"""

Q_POINTS = """
PREFIX ex: <https://example.org/taltech-edt/>
SELECT (COUNT(DISTINCT ?point) AS ?count)
WHERE {
    ?point ex:variableName ?variable .
}
"""

Q_TRACEABLE = """
PREFIX ex: <https://example.org/taltech-edt/>
PREFIX sosa: <http://www.w3.org/ns/sosa/>

SELECT (COUNT(?observation) AS ?count)
WHERE {
    ?observation
        a sosa:Observation ;
        ex:observedBySemanticPoint ?point ;
        ex:sourceOperationalDevice ?device ;
        ex:sourceOperationalUUID ?uuid ;
        sosa:resultTime ?timestamp ;
        sosa:hasSimpleResult ?value .

    ?point
        ex:representsMeasurementFrom ?device ;
        ex:variableName ?variable .
}
"""

Q_PROVENANCE = """
PREFIX ex: <https://example.org/taltech-edt/>
PREFIX sosa: <http://www.w3.org/ns/sosa/>
PREFIX prov: <http://www.w3.org/ns/prov#>

SELECT (COUNT(?observation) AS ?count)
WHERE {
    ?observation
        a sosa:Observation ;
        ex:sourceOperationalUUID ?uuid ;
        prov:wasDerivedFrom ?source .
}
"""

Q_SPATIAL = """
PREFIX ex: <https://example.org/taltech-edt/>

SELECT (COUNT(?point) AS ?count)
WHERE {
    ?point ex:reconciledSpatialContext ?space .
}
"""

print("\n====================================")
print("COMPACT SPARQL VALIDATION")
print("====================================")

n_obs = scalar_count(
    "Q1 observations",
    Q_OBS
)

n_devices = scalar_count(
    "Q2 devices",
    Q_DEVICES
)

n_points = scalar_count(
    "Q3 semantic points",
    Q_POINTS
)

n_traceable = scalar_count(
    "Q4 traceable observations",
    Q_TRACEABLE
)

n_prov = scalar_count(
    "Q5 provenance",
    Q_PROVENANCE
)

n_spatial = scalar_count(
    "Q6 spatial mappings",
    Q_SPATIAL
)

# ============================================================
# 6. DEVICE × VARIABLE COUNTS
# ============================================================

Q_COUNTS = """
PREFIX ex: <https://example.org/taltech-edt/>
PREFIX sosa: <http://www.w3.org/ns/sosa/>

SELECT
    ?uuid
    ?variable
    (COUNT(?observation) AS ?observation_count)

WHERE {
    ?observation
        a sosa:Observation ;
        ex:sourceOperationalUUID ?uuid ;
        ex:observedBySemanticPoint ?point .

    ?point
        ex:variableName ?variable .
}

GROUP BY
    ?uuid
    ?variable

ORDER BY
    ?uuid
    ?variable
"""

start = time.perf_counter()

counts_rows = []

for row in store.query(Q_COUNTS):

    counts_rows.append({
        "uuid":
            str(term_to_python(row[0])),

        "variable":
            str(term_to_python(row[1])),

        "observation_count":
            int(term_to_python(row[2]))
    })

elapsed = time.perf_counter() - start

query_timings.append({
    "query":
        "Q7 counts by device-variable",

    "seconds":
        elapsed,

    "rows_returned":
        len(counts_rows)
})

counts_df = pd.DataFrame(
    counts_rows
)

counts_df.to_csv(
    OUT /
    "01_observation_counts_by_device_variable.csv",
    index=False
)

print("\n====================================")
print("DEVICE × VARIABLE COUNTS")
print("====================================")

print(
    counts_df.to_string(
        index=False
    )
)

print(
    f"Q7 time: {elapsed:.3f} s"
)

# ============================================================
# 7. TRACEABILITY VALIDATION
# ============================================================

EXPECTED_OBS = 323_266
EXPECTED_DEVICES = 7
EXPECTED_POINTS = 21
EXPECTED_SPATIAL = 0

checks = pd.DataFrame(
    [
        {
            "check":
                "RDF observations",

            "expected":
                EXPECTED_OBS,

            "actual":
                n_obs
        },

        {
            "check":
                "Operational devices",

            "expected":
                EXPECTED_DEVICES,

            "actual":
                n_devices
        },

        {
            "check":
                "Semantic points",

            "expected":
                EXPECTED_POINTS,

            "actual":
                n_points
        },

        {
            "check":
                "Traceable observations",

            "expected":
                EXPECTED_OBS,

            "actual":
                n_traceable
        },

        {
            "check":
                "Observations with provenance",

            "expected":
                EXPECTED_OBS,

            "actual":
                n_prov
        },

        {
            "check":
                "Native spatial mappings",

            "expected":
                EXPECTED_SPATIAL,

            "actual":
                n_spatial
        }
    ]
)

checks[
    "passed"
] = (
    checks["expected"]
    ==
    checks["actual"]
)

checks.to_csv(
    OUT /
    "02_native_traceability_checks.csv",
    index=False
)

print("\n====================================")
print("TRACEABILITY VALIDATION")
print("====================================")

print(
    checks.to_string(
        index=False
    )
)

assert checks["passed"].all(), (
    "At least one traceability check failed."
)

print(
    "\n✓ All traceability checks passed"
)

# ============================================================
# 8. STREAM FULL ML EXTRACTION DIRECTLY TO CSV
# ============================================================

# The full query returns 323,266 rows.
# We do NOT accumulate them in a Python list.
# Each row is written immediately to disk.

ML_QUERY = """
PREFIX ex: <https://example.org/taltech-edt/>
PREFIX sosa: <http://www.w3.org/ns/sosa/>

SELECT
    ?uuid
    ?timestamp
    ?variable
    ?value
    ?unit
    ?point
    ?observation

WHERE {
    ?observation
        a sosa:Observation ;
        ex:sourceOperationalUUID ?uuid ;
        ex:observedBySemanticPoint ?point ;
        sosa:resultTime ?timestamp ;
        sosa:hasSimpleResult ?value ;
        ex:sourceUnit ?unit .

    ?point
        ex:variableName ?variable .
}
"""

ML_FILE = (
    OUT /
    "03_ml_extraction_long_traceable.csv"
)

print("\n====================================")
print("STREAMING ML EXTRACTION")
print("====================================")

start = time.perf_counter()
row_count = 0

with ML_FILE.open(
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.writer(f)

    writer.writerow(
        [
            "uuid",
            "timestamp",
            "variable",
            "value",
            "unit",
            "point",
            "observation"
        ]
    )

    for row in store.query(
        ML_QUERY
    ):

        writer.writerow(
            [
                term_to_python(
                    row[0]
                ),

                term_to_python(
                    row[1]
                ),

                term_to_python(
                    row[2]
                ),

                term_to_python(
                    row[3]
                ),

                term_to_python(
                    row[4]
                ),

                term_to_python(
                    row[5]
                ),

                term_to_python(
                    row[6]
                )
            ]
        )

        row_count += 1

elapsed = time.perf_counter() - start

query_timings.append({
    "query":
        "Q8 full ML extraction",

    "seconds":
        elapsed,

    "rows_returned":
        row_count
})

print(
    f"ML rows : {row_count:,}"
)

print(
    f"Time    : {elapsed:.2f} s"
)

print(
    "Saved   :",
    ML_FILE
)

assert row_count == EXPECTED_OBS, (
    f"Expected {EXPECTED_OBS:,} ML rows "
    f"but extracted {row_count:,}"
)

print(
    "✓ Full ML extraction preserves "
    "all observations"
)

# ============================================================
# 9. SAVE PERFORMANCE TABLE
# ============================================================

timings_df = pd.DataFrame(
    query_timings
)

timings_df.to_csv(
    OUT /
    "04_sparql_query_timings.csv",
    index=False
)

load_df = pd.DataFrame(
    [
        {
            "graph_file":
                GRAPH_FILE.name,

            "file_size_mb":
                round(
                    GRAPH_FILE.stat().st_size /
                    (1024**2),
                    3
                ),

            "load_seconds_first_import":
                load_seconds,

            "backend":
                "PyOxigraph persistent disk store"
        }
    ]
)

load_df.to_csv(
    OUT /
    "05_rdf_load_performance.csv",
    index=False
)

print("\n====================================")
print("SPARQL PERFORMANCE")
print("====================================")

print(
    timings_df.to_string(
        index=False
    )
)

print("\n====================================")
print("FILES CREATED")
print("====================================")

for f in sorted(
    OUT.iterdir()
):
    print(
        " -",
        f.name
    )

print(
    "\n✓ G_native SPARQL validation "
    "completed successfully"
)
